<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎨 Qwen-Image 2.1 Uncensored - Create, Edit & Multi-Reference</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab T4 Edition - Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0 0 0 0;'>Text-to-Image · Image Editing · Up to 10 Reference Images · Inpainting · Outpainting · Transparent PNGs · INT8 Tensor Cores · Turbo LoRAs (Viggle 6-Step, Pruna 8-Step, Alibaba 4-Step) | Powered by Wan2GP + mmgp</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/AIQUESTAcademy-blueviolet?style=for-the-badge&logo=youtube&logoColor=white" />
  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>

</div>

---

### What is this notebook?

Image generation and editing with **Qwen-Image 2.1 Uncensored** ([abenzerps/Qwen-Image-2.1-Uncensored-GGUF](https://huggingface.co/abenzerps/Qwen-Image-2.1-Uncensored-GGUF), the `int8_convrot` file), running on the **Wan2GP** engine.

- **Three tasks:** 🖼️ **Create** from text · 🧩 **Edit & Multi-Reference** (edit one image with an instruction, or combine up to 10 reference images: people, objects, clothing, scenes) · 🖌️ **Inpaint** (paint the area to change) · 🔲 **Outpaint** (extend the image to any side or to a target aspect ratio)
- **🫥 Transparent images:** one checkbox for RGBA output (stickers, logos, cut-outs, subject extraction), saved as PNG with an alpha channel
- **INT8 tensor cores:** the 7B transformer's int8 weights run on real INT8 matmul kernels (Comfy Kitchen), so they aren't converted to FP16 on every step
- **FP16 on T4:** the transformer runs in FP16 (the T4 has no BF16 tensor cores) and stays in VRAM between images
- **Three speed modes:** ✨ **Turbo HQ** (Viggle v0.2.1 LoRA, 6 steps, close to the base model in quality, the default) · ⚡ **Turbo** (Pruna v0.1 LoRA, 8 steps) · 🚀 **Turbo Fast** (Alibaba PAI's official Fun-Acc LoRA, 4 steps) · 🎨 **Quality** (full 40-step base model with CFG and a negative prompt)
- **Qwen3-VL-8B text encoder:** INT8, runs once per new prompt (prompts are cached), then leaves the GPU

### Quick Start
1. **Runtime → Change runtime type → T4 GPU**
2. Run **Step 1** (install + ~19 GB of downloads, a few minutes)
3. Run **Step 2**: it loads the model (~1 min), then prints the **Cloudflare link** (`*.trycloudflare.com`) and **Gradio link** when everything is ready
4. For best prompt adherence use **Quality** mode, describe what is visible, and put exact text in "double quotes"
5. Images are also saved to `/content/outputs`

In [1]:
#@title 📦 Step 1: GPU Check, Install, Wan2GP & Download Models
# Installs only the packages the Qwen-Image 2.1 pipeline imports, fetches Wan2GP at a pinned commit and
# downloads the uncensored INT8 transformer + Qwen3-VL-8B INT8 text encoder + VAE + Turbo LoRAs (Viggle, Pruna, Alibaba).
# Safe to re-run: finished downloads are skipped.
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"  # no duplicate chunk cache on disk
os.environ.pop("HF_XET_HIGH_PERFORMANCE", None)    # its large transfer buffers exhaust Colab's 12.7 GB RAM

import codecs, fcntl, pty, shutil, stat, struct, subprocess, sys, termios, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
WAN2GP_DIR = os.path.join(ROOT, "Wan2GP")
CKPT_DIR = os.path.join(ROOT, "qwen21_ckpts")  # outside the repo, so a fresh checkout never touches the weights
WAN2GP_COMMIT = "91301f0eda923b6c4b05159e32f1dfd2a960af9d"  # version this notebook is tuned on (27 Sep 2026)
CLOUDFLARED = "/tmp/cloudflared"
t0 = time.time()

# ── 1. GPU / RAM check ──
import psutil, torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU detected. Go to Runtime → Change runtime type → T4 GPU, then run this cell again.")
props = torch.cuda.get_device_properties(0)
print(f"✅ GPU: {props.name} · {props.total_memory / 1024**3:.1f} GB VRAM · SM {props.major}.{props.minor}")
print(f"✅ RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB · Disk free: {shutil.disk_usage(ROOT).free / 1024**3:.0f} GB")
if (props.major, props.minor) < (7, 5):
    print("⚠️ INT8 tensor-core kernels need a T4 (SM 7.5) or newer; this GPU will use the slower PyTorch INT8 path.")

# ── 2. Python packages: only what the Qwen-Image 2.1 pipeline imports (Colab's torch is kept) ──
print("\n📦 Installing packages...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", "--timeout", "120", "--retries", "5",
    "mmgp==3.8.2",            # Wan2GP's offloading engine (version pinned by this Wan2GP commit)
    "comfy-kitchen==0.2.35",  # INT8 tensor-core kernels (T4 supported)
    "transformers==4.54.0",   # version Wan2GP pins (tokenizer / processor)
    "diffusers==0.36.0",      # scheduler + model base classes
    "gradio==5.50.0",
    "optimum-quanto", "accelerate", "einops", "ffmpeg-python", "hf_xet",
], check=True)
print("✅ Packages ready")

# ── 3. Wan2GP source at a pinned commit (a fresh checkout also wipes old runtime patches) ──
print("\n📥 Fetching Wan2GP...")
def git(*args):
    subprocess.run(["git", "-C", WAN2GP_DIR, *args], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if not os.path.isdir(os.path.join(WAN2GP_DIR, ".git")):
    shutil.rmtree(WAN2GP_DIR, ignore_errors=True)
    os.makedirs(WAN2GP_DIR)
    git("init", "-q")
    git("remote", "add", "origin", "https://github.com/DeepBeepMeep/Wan2GP.git")
git("fetch", "-q", "--depth", "1", "origin", WAN2GP_COMMIT)
git("checkout", "-q", "-f", "FETCH_HEAD")
git("reset", "-q", "--hard")
git("clean", "-q", "-fd")
assert os.path.exists(os.path.join(WAN2GP_DIR, "models/qwen21/qwen21_handler.py")), "Wan2GP checkout is incomplete"
print(f"✅ Wan2GP @ {WAN2GP_COMMIT[:10]}")

# ── 4. Model weights + cloudflared, downloaded in parallel ──
from huggingface_hub import hf_hub_download

UNCENSORED_REPO = "abenzerps/Qwen-Image-2.1-Uncensored-GGUF"
QWEN_REPO = "DeepBeepMeep/Qwen_image_2"   # VAE, scheduler, Pruna LoRA (Wan2GP's own Qwen-Image 2.1 files)
ENCODER_REPO = "DeepBeepMeep/Ideogram4"   # Qwen3-VL-8B text encoder in Wan2GP's layout
VIGGLE_REPO = "Viggle/Qwen-Image-2.1-viggle-turbo"  # Viggle Turbo v0.2.1 (rank 128 = same images as rank 256)
ALIBABA_REPO = "alibaba-pai/Qwen-Image-2.1-Fun-Acc-LoRAs"  # Alibaba PAI Fun-Acc 4-step (PDD distillation)
TE = "Qwen3-VL-8B-Instruct"
TE_FILES = ["config_legacy.json", "tokenizer_legacy.json", "tokenizer_config_legacy.json", "preprocessor_config.json",
            "chat_template.jinja", "merges.txt", "vocab.json", "added_tokens.json", "special_tokens_map.json",
            "video_preprocessor_config.json"]
FILES = [
    (UNCENSORED_REPO, "qwen-image-2.1-UC-int8_convrot.safetensors"),     # 7.3 GB uncensored transformer
    (ENCODER_REPO, f"{TE}/{TE}_int8_convrot.safetensors"),             # 8.9 GB text encoder
    *[(ENCODER_REPO, f"{TE}/{name}") for name in TE_FILES],
    (QWEN_REPO, "qwen_image_21/qwen_image_21_vae.safetensors"),         # 1.4 GB VAE
    (QWEN_REPO, "qwen_image_21/vae_config.json"),
    (QWEN_REPO, "qwen_image_21/scheduler_config.json"),
    (VIGGLE_REPO, "Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors"),  # 0.7 GB Viggle Turbo HQ LoRA
    (QWEN_REPO, "loras/p_qwen_image_2.1_8step_v0.1.safetensors"),       # 0.3 GB Pruna Turbo 8-step LoRA
    (ALIBABA_REPO, "models/Qwen-Image-2.1-Fun-Acc-4Step.safetensors"),  # 0.35 GB Alibaba 4-step LoRA + heads
    (ALIBABA_REPO, "models/pdd_config.json"),                           # its fixed 4-step sigma schedule
]

def fetch_small(repo, filename):
    hf_hub_download(repo_id=repo, filename=filename, local_dir=CKPT_DIR)
    return filename

def fetch_large(repo, filename):
    # One big file at a time, each in its own process: download buffers never pile up in the notebook's RAM
    # and are fully returned to the system when the file is done
    code = ("import sys; from huggingface_hub import hf_hub_download; "
            "hf_hub_download(repo_id=sys.argv[1], filename=sys.argv[2], local_dir=sys.argv[3])")
    t = time.time()
    # Colab only displays the notebook's own output, and huggingface_hub hides its progress bar when not on a
    # terminal: run the child on a pseudo-terminal and relay its tqdm bar into the cell
    master, slave = pty.openpty()
    fcntl.ioctl(slave, termios.TIOCSWINSZ, struct.pack("HHHH", 24, 100, 0, 0))  # a new pty is 0 columns wide
    proc = subprocess.Popen([sys.executable, "-c", code, repo, filename, CKPT_DIR],
                            stdout=slave, stderr=slave, close_fds=True)
    os.close(slave)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")  # bar glyphs can span read chunks
    while True:
        try:
            chunk = os.read(master, 4096)
        except OSError:  # EIO: the child closed the terminal
            break
        if not chunk:
            break
        sys.stdout.write(decoder.decode(chunk).replace("\r\n", "\n"))
        sys.stdout.flush()
    os.close(master)
    if proc.wait() != 0:
        raise RuntimeError(f"download exited with code {proc.returncode}")
    print()
    size = os.path.getsize(os.path.join(CKPT_DIR, filename)) / 1e9  # decimal GB, as listed on Hugging Face
    print(f"✅ {filename} ({size:.1f} GB, {time.time() - t:.0f}s)")

def get_cloudflared():
    if not os.path.exists(CLOUDFLARED):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CLOUDFLARED)
        os.chmod(CLOUDFLARED, os.stat(CLOUDFLARED).st_mode | stat.S_IEXEC)
    return "cloudflared"

print("\n⬇️ Downloading configs, tokenizer and cloudflared...")
small = [(repo, name) for repo, name in FILES if not name.endswith(".safetensors")]
failed = False
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = [pool.submit(fetch_small, repo, name) for repo, name in small] + [pool.submit(get_cloudflared)]
    for fut in futures:
        try:
            fut.result()
        except Exception as e:
            failed = True
            print(f"❌ Download failed: {e}")

print("⬇️ Downloading ~19 GB of weights (one file at a time to stay within Colab's RAM)...")
for repo, name in FILES:
    if name.endswith(".safetensors"):
        try:
            fetch_large(repo, name)
        except Exception as e:
            failed = True
            print(f"❌ Download failed: {name}: {e}")
if failed:
    raise SystemExit("❌ Some downloads failed. Run this cell again (finished files are skipped).")

print(f"\n💾 Disk free: {shutil.disk_usage(ROOT).free / 1024**3:.0f} GB")
print(f"🎉 Step 1 done in {(time.time() - t0) / 60:.1f} min. Run Step 2 to launch the generator.")

✅ GPU: Tesla T4 · 14.6 GB VRAM · SM 7.5
✅ RAM: 12.7 GB · Disk free: 65 GB

📦 Installing packages...
✅ Packages ready

📥 Fetching Wan2GP...
✅ Wan2GP @ 91301f0eda

⬇️ Downloading configs, tokenizer and cloudflared...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config_legacy.json: 0.00B [00:00, ?B/s]

config_legacy.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

...

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

vae_config.json: 0.00B [00:00, ?B/s]

scheduler_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

pdd_config.json: 0.00B [00:00, ?B/s]

⬇️ Downloading ~19 GB of weights (one file at a time to stay within Colab's RAM)...
qwen-image-2.1-UC-int8_convrot.safetenso(…): 100%|██████████████| 7.26G/7.26G [00:40<00:00, 178MB/s]

✅ qwen-image-2.1-UC-int8_convrot.safetensors (7.3 GB, 41s)
Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruc(…): 100%|██████████████| 8.94G/8.94G [00:52<00:00, 169MB/s]

✅ Qwen3-VL-8B-Instruct/Qwen3-VL-8B-Instruct_int8_convrot.safetensors (8.9 GB, 54s)
qwen_image_21/qwen_image_21_vae.safetens(…): 100%|██████████████| 1.35G/1.35G [00:11<00:00, 119MB/s]

✅ qwen_image_21/qwen_image_21_vae.safetensors (1.4 GB, 12s)
Qwen-Image-2.1-viggle-turbo-v0.2.1-6step(…): 100%|███████████████| 680M/680M [00:11<00:00, 61.5MB/s]

✅ Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors (0.7 GB, 12s)
loras/p_qwen_image_2.1_8step_v0.1.safete(…): 100%|███████████████| 336M/336M [00:05<00:00, 60.3MB/s]

✅ loras/p_qwen_image_2.1_8step_v0.1.safetensors (0.3 GB, 6s)
models/Qwen-Image-2.1-Fun-Acc-4Step.safe(…): 100%|██████████████

In [ ]:
#@title 🚀 Step 2: Load Qwen-Image 2.1 Engine & Launch Gradio
# Loads the model (INT8 ConvRot weights on Comfy Kitchen INT8 tensor-core kernels, FP16 transformer kept in VRAM),
# then prints the public links once everything is ready. Re-running this cell reuses the loaded model.

import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Keep diffusers/transformers from importing Colab's JAX/Flax and TensorFlow (deprecation spam, slower start)
os.environ["USE_FLAX"] = "0"
os.environ["USE_TF"] = "0"

import contextlib, gc, io, random, re, stat, subprocess, sys, threading, time, traceback, types, urllib.request, warnings, zipfile
from types import SimpleNamespace
warnings.filterwarnings("ignore", message=".*parameter in the Blocks constructor.*")

ROOT = "/content" if os.path.isdir("/content") else os.getcwd()
WAN2GP_DIR = os.path.join(ROOT, "Wan2GP")
CKPT_DIR = os.path.join(ROOT, "qwen21_ckpts")
OUT_DIR = os.path.join(ROOT, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir(os.path.join(WAN2GP_DIR, "models", "qwen21")):
    raise SystemExit("❌ Wan2GP not found. Run Step 1 first.")
if WAN2GP_DIR not in sys.path:
    sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)

# shared/utils/utils.py imports rembg (background removal + onnxruntime) at module level. Text-to-image never
# calls it, so a stub replaces the whole dependency.
if "rembg" not in sys.modules:
    _rembg = types.ModuleType("rembg")
    def _no_rembg(*args, **kwargs):
        raise RuntimeError("Background removal is not available in this notebook.")
    _rembg.remove = _rembg.new_session = _no_rembg
    sys.modules["rembg"] = _rembg

import numpy as np
import psutil
import torch
import gradio as gr
from PIL import Image
from mmgp import offload, quant_router
from optimum.quanto.tensor.weights import qbytes
from shared.utils import files_locator as fl
from shared.kernels import int8_backend, quanto_int8_inject

# Wan2GP's UI (wgp.py) registers the INT8 ConvRot loader at startup; this notebook imports the model directly
quant_router.unregister_handler(".fp8_quanto_bridge")
quant_router.register_handler("shared.qtypes.int8_convrot")
fl.set_checkpoints_paths([CKPT_DIR])
from models.qwen21.qwen21_handler import family_handler, ENCODER

BASE_MODEL_TYPE = "qwen_image_21_7B"
TRANSFORMER_FILE = os.path.join(CKPT_DIR, "qwen-image-2.1-UC-int8_convrot.safetensors")
TEXT_ENCODER_FILE = os.path.join(CKPT_DIR, ENCODER, f"{ENCODER}_int8_convrot.safetensors")
VIGGLE_LORA_FILE = os.path.join(CKPT_DIR, "Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors")
PRUNA_LORA_FILE = os.path.join(CKPT_DIR, "loras", "p_qwen_image_2.1_8step_v0.1.safetensors")
ALIBABA_LORA_FILE = os.path.join(CKPT_DIR, "models", "Qwen-Image-2.1-Fun-Acc-4Step.safetensors")
ALIBABA_CONFIG_FILE = os.path.join(CKPT_DIR, "models", "pdd_config.json")

# ==== GPU INFO ====
gpu = torch.cuda.get_device_properties(0)
# T4 (SM 7.5) has no BF16 tensor cores: the transformer runs in FP16 there (BF16 attention would fall back to the
# slow math kernel). Ampere+ uses native BF16. The text encoder stays BF16 (LLM activations can overflow FP16).
DTYPE = torch.bfloat16 if gpu.major >= 8 else torch.float16
INT8_LABELS = {"kitchen": "Comfy Kitchen INT8", "triton": "Triton INT8", "pytorch": "PyTorch (exact)"}
FP16_MAX = 65504.0
PHASE_TIMES = {}
ACT_PEAK = {}
KV_MARGIN_BYTES = int(1.5 * 1024**3)  # VRAM kept for denoising activations (the cache is freed before the VAE runs)
VAE_TILE = 1024  # 1 tile up to 1024px (Wan2GP's auto setting picks 512px tiles below 16 GB: 9 tiles at 1024px)


def _gb(n):
    return f"{n / 1024**3:.1f} GB"


def _dt(dtype):
    return str(dtype).replace("torch.", "").upper()


class CPUEmbedding(torch.nn.Module):
    """Token lookup from the memory-mapped table on the CPU. Hidden from mmgp (no Parameter), so the 1.2 GB
    table is no longer streamed to the GPU for every new prompt and the rest of the encoder fits in the RAM
    page cache (new prompts then encode in seconds instead of re-reading ~8 GB from disk)."""

    def __init__(self, weight):
        super().__init__()
        self._table = [weight]

    def forward(self, input_ids):
        return torch.nn.functional.embedding(input_ids.cpu(), self._table[0]).to(input_ids.device)


def _load_engine():
    for path in (TRANSFORMER_FILE, TEXT_ENCODER_FILE, VIGGLE_LORA_FILE, PRUNA_LORA_FILE, ALIBABA_LORA_FILE, ALIBABA_CONFIG_FILE):
        if not os.path.exists(path):
            raise SystemExit(f"❌ {path} not found. Run Step 1 first.")

    # ---- INT8 tensor-core kernels (Wan2GP's "INT8 Kernels: Auto") ----
    # Without this every INT8 layer is dequantized and run as an FP16 matmul. Auto picks Comfy Kitchen,
    # then Triton, then plain PyTorch.
    try:
        int8_backend.configure("auto", 1)
    except Exception as e:
        print(f"⚠️ INT8 kernels unavailable, using the PyTorch INT8 path: {e}")
    # Wan2GP splits each INT8 matmul into 16 MB row tiles; 384 MB runs a whole 1024px image per matmul
    int8_backend._SCRATCH_BYTES = 384 * 1024 * 1024

    # ---- Load transformer + text encoder + VAE through Wan2GP's Qwen-Image 2.1 handler ----
    # The handler always loads the transformer as BF16 (the T4 can only emulate it): ask for DTYPE instead.
    # The VAE passes its own FP32 dtype and the text encoder uses a different loader, so both stay unchanged.
    print(f"📂 Loading Qwen-Image 2.1 Uncensored (INT8 ConvRot) · transformer {_dt(DTYPE)}...")
    _fast_load = offload.fast_load_transformers_model
    def _fast_load_dtype(*args, **kwargs):
        kwargs.setdefault("default_dtype", DTYPE)
        return _fast_load(*args, **kwargs)
    offload.fast_load_transformers_model = _fast_load_dtype
    try:
        model_def = family_handler.query_model_def(BASE_MODEL_TYPE, {})
        pipeline, parts = family_handler.load_model([TRANSFORMER_FILE], BASE_MODEL_TYPE, BASE_MODEL_TYPE, model_def,
                                                    text_encoder_filename=TEXT_ENCODER_FILE)
    finally:
        offload.fast_load_transformers_model = _fast_load
    transformer = parts["transformer"]
    text_model = parts["text_encoder"]
    text_model.embed_tokens = CPUEmbedding(text_model.embed_tokens.weight.data)

    # ---- Memory plan for Colab (15 GB VRAM, ~12.7 GB RAM) ----
    # Transformer (7.3 GB) and VAE (1.4 GB) stay in VRAM for good (co-tenants "*" are never swapped out).
    # The 8B text encoder streams through a 2 GB budget only when a new prompt appears, then leaves the GPU;
    # the Qwen3-VL vision tower (~0.6 GB, reference images / edits only) loads the same way.
    # No pinned RAM: weights are read straight from the memory-mapped files.
    offload_obj = offload.profile(
        {"transformer": transformer, "text_encoder": text_model, "vision_encoder": parts["vision_encoder"],
         "vae": parts["vae"]},
        profile_no=4,
        pinnedMemory=False,
        quantizeTransformer=False,
        extraModelsToQuantize=[],
        budgets={"transformer": 0, "vae": 0, "vision_encoder": 0, "text_encoder": 2000},
        coTenantsMap={"transformer": "*", "vae": "*"},
        loras=["transformer"],  # hooks the transformer's linears so the Turbo LoRA can be attached
        convertWeightsFloatTo=DTYPE,
        asyncTransfers=True,
    )
    offload.shared_state["_attention"] = "sdpa"  # memory-efficient SDPA (FlashAttention needs SM 8.0+)

    # ---- Turbo LoRAs: Viggle v0.2.1 (6 steps), Pruna v0.1 (8 steps), Alibaba Fun-Acc (4 steps). All stay active:
    # mmgp only copies active LoRAs to the GPU when the transformer loads, so each mode sets the others to 0. ----
    def lora_keys(sd):
        # Alibaba's PDD export names its adapters "<layer>.lora_down/.lora_up" (alpha = rank, so strength 1.0);
        # its output heads and norm weights are not LoRA deltas and are applied separately below
        if any(key.endswith(".lora_down") for key in sd):
            sd = {"transformer." + key.rsplit(".", 1)[0] + (".lora_A.weight" if key.endswith(".lora_down") else ".lora_B.weight"): value
                  for key, value in sd.items() if key.endswith((".lora_down", ".lora_up"))}
        return transformer.preprocess_loras(BASE_MODEL_TYPE, sd)

    loaded = offload.load_loras_into_model(
        transformer, [VIGGLE_LORA_FILE, PRUNA_LORA_FILE, ALIBABA_LORA_FILE], [1.0, 1.0, 1.0], activate_all_loras=True,
        pinnedLora=False,
        preprocess_sd=lora_keys,
        split_linear_modules_map=getattr(transformer, "split_linear_modules_map", None),
    )
    adapters = {path: str(no) for no, path in enumerate(loaded)}  # mmgp numbers only the LoRAs that loaded
    for error in getattr(transformer, "_loras_errors", None) or []:
        print(f"⚠️ LoRA failed to load: {error}")

    # ---- FP16 headroom monitor: each block clips its output at ±65504; a peak there means overflow ----
    if DTYPE == torch.float16:
        def track_peak(module, args, output):
            peak = (output[0] if isinstance(output, tuple) else output).detach().abs().amax().float()
            ACT_PEAK["fp16"] = peak if "fp16" not in ACT_PEAK else torch.maximum(ACT_PEAK["fp16"], peak)
        for block in transformer.transformer_blocks:
            block.register_forward_hook(track_peak)

    # ---- Text encoder: exact INT8 path, then off the GPU ----
    # INT8 activation quantization is approximate and LLM activations have outliers; the encoder only runs
    # for new prompts (results are cached), so the exact path costs nothing per step.
    encode_fast = pipeline._encode_conditioning

    def _to_dit_dtype(embeds):
        # txt_in starts with a per-token RMSNorm, which ignores each token's scale: shrink any token whose
        # LLM features would overflow FP16 (exact, unlike clamping)
        if DTYPE == torch.float16:
            peak = embeds.float().abs().amax(dim=-1, keepdim=True)
            embeds = embeds.float() / (peak / 32768).clamp(min=1)
        return embeds.to(DTYPE)

    def encode_exact(prompts, images, device, batch_size):
        t0 = time.time()
        backend = int8_backend._backend
        fast_forward = qbytes.WeightQBytesLinearFunction.__dict__["forward"]
        int8_backend._backend = "pytorch"
        qbytes.WeightQBytesLinearFunction.forward = staticmethod(quanto_int8_inject._default_quanto_qbytes_linear_forward)
        try:
            contexts = encode_fast(prompts, images, device, batch_size)
        finally:
            int8_backend._backend = backend
            qbytes.WeightQBytesLinearFunction.forward = fast_forward
            if set(offload_obj.active_models_ids) - {"transformer", "vae"}:  # text / vision encoder
                offload_obj.unload_all(keep=["transformer", "vae"])
            PHASE_TIMES["text"] = time.time() - t0
        return [(_to_dit_dtype(embeds), mask, slots) for embeds, mask, slots in contexts]

    pipeline._encode_conditioning = encode_exact

    # ---- VAE decode: timing, NaN guard (non-finite latents decode to black), 512px tiles if 1024 runs out ----
    decode_fast = pipeline.vae.decode_to_cpu_uint8

    def decode_checked(latent_holder, **kwargs):
        t0 = time.time()
        latents = latent_holder[0]
        if not torch.isfinite(latents).all():
            raise RuntimeError("non-finite latents")
        try:
            return decode_fast(latent_holder, **kwargs)
        except torch.OutOfMemoryError:
            print("⚠️ VAE ran out of VRAM with 1024px tiles - retrying with 512px tiles.")
            gc.collect(); torch.cuda.empty_cache()
            pipeline.vae.configure_tiling((True, 512), latents.shape[-1] * 16, latents.shape[-2] * 16)
            return decode_fast([latents], **kwargs)
        finally:
            PHASE_TIMES["vae"] = time.time() - t0

    pipeline.vae.decode_to_cpu_uint8 = decode_checked

    # ---- Move the transformer + VAE into VRAM now, so the first image doesn't pay for it ----
    for model_id in ("transformer", "vae"):
        offload_obj.ensure_model_loaded(model_id)
    pdd = load_alibaba_pdd(transformer) if adapters.get(ALIBABA_LORA_FILE) is not None else None
    return SimpleNamespace(pipeline=pipeline, transformer=transformer, offload=offload_obj, dtype=DTYPE, pdd=pdd,
                           adapters={"viggle": adapters.get(VIGGLE_LORA_FILE), "pruna": adapters.get(PRUNA_LORA_FILE),
                                     "alibaba": adapters.get(ALIBABA_LORA_FILE)})


def load_alibaba_pdd(transformer):
    """Alibaba Fun-Acc is Parallel Decoding Distillation, not just a LoRA: step k of 4 uses its own output head
    (proj_out), 65 norm weights are replaced, and it samples on a fixed, already-shifted sigma grid. The heads and
    norms are swapped in only while this mode runs; the grid goes through Wan2GP's fixed-sigma path (the one the
    Pruna mode uses: no dynamic shift, no terminal stretch; Pruna is registered for 8 steps, this one for 4)."""
    import json
    from safetensors import safe_open
    import models.qwen21.pipeline as qwen_pipeline

    with open(ALIBABA_CONFIG_FILE, encoding="utf-8") as reader:
        config = json.load(reader)
    sigmas = [float(value) for value in config["pdd_sigmas"]]
    if config.get("pdd_block_size") != 1 or len(sigmas) != config["pdd_num_steps"] + 1:
        raise ValueError(f"Unsupported Alibaba PDD export: {config.get('pdd_export_format')}")
    qwen_pipeline.PRUNA_SIGMAS[len(sigmas) - 1] = tuple(sigmas[:-1])  # the scheduler appends the final 0
    head = transformer.proj_out.weight
    with safe_open(ALIBABA_LORA_FILE, framework="pt", device="cpu") as reader:
        heads = reader.get_tensor("proj_out.weight").to(head.device, head.dtype)
        norms = []
        for name in config["pdd_full_parameters"]:
            param = transformer.get_parameter(name)
            norms.append((param, param.data.clone(), reader.get_tensor(name).to(param.device, param.dtype)))
    pdd = SimpleNamespace(active=False, heads=heads, base_head=head.data.clone(), norms=norms,
                          steps=len(sigmas) - 1, sigmas=sigmas)

    def select_head(module, args):
        if pdd.active:  # Wan2GP records the absolute step index on the transformer before every step
            module.weight.data.copy_(pdd.heads[min(getattr(transformer, "_lora_step_no", 0), pdd.steps - 1)])

    transformer.proj_out.register_forward_pre_hook(select_head)
    return pdd


def set_alibaba_pdd(engine, active):
    pdd = engine.pdd
    if pdd is None or pdd.active == active:
        return
    for param, base, replacement in pdd.norms:
        param.data.copy_(replacement if active else base)
    if not active:
        engine.transformer.proj_out.weight.data.copy_(pdd.base_head)
    pdd.active = active


def run_generation(prompt, negative_prompt, mode, steps, guidance, width, height, seed, callback=None,
                   ref_images=None, edit_image=None, edit_mask=None, strength=1.0, outpainting_dims=None, rgba=False):
    """One image. ref_images: PIL references (<image1>, <image2>...). edit_image + edit_mask: masked edit of an image
    already sized width x height (mask white = change). If the INT8 kernels ever produce NaN/Inf, retry on the
    exact INT8 path."""
    engine = QWEN_ENGINE
    PHASE_TIMES.clear()
    ACT_PEAK.clear()
    spec = MODES[mode]
    names = [no for no in engine.adapters.values() if no is not None]
    offload.activate_loras(engine.transformer, names,
                           [1.0 if no == engine.adapters.get(spec["lora"]) else 0.0 for no in names])
    set_alibaba_pdd(engine, spec["lora"] == "alibaba")
    if spec["lora"] == "alibaba":
        print(f"Alibaba Fun-Acc: fixed sigmas {[round(s, 4) for s in engine.pdd.sigmas]} + a separate output head per step "
              "(Wan2GP logs this fixed-sigma path as 'Pruna Scheduler')")
    turbo = spec["lora"] is not None
    # KV cache: text + reference/edit-image tokens never change between steps (they are modulated at t=0), so their
    # attention keys/values are computed once and later steps only run the output image's tokens. Enabled when the
    # cache (K+V for every block, per CFG branch) fits in free VRAM next to the working memory.
    images = list(ref_images or []) + ([edit_image] if edit_image is not None else [])
    text_tokens = max(len(engine.pipeline.tokenizer.encode(text or " ")) for text in (prompt, negative_prompt)) + 64
    prefix_tokens = text_tokens + sum((image.width // 16) * (image.height // 16) for image in images)
    cache_bytes = prefix_tokens * 2 * 4096 * 2 * len(engine.transformer.transformer_blocks) * (1 if turbo else 2)
    free = torch.cuda.mem_get_info()[0] + torch.cuda.memory_reserved() - torch.cuda.memory_allocated()
    use_kv = cache_bytes < free - KV_MARGIN_BYTES
    PHASE_TIMES["kv"] = f"KV cache {cache_bytes / 1024**3:.1f} GB" if use_kv else "KV cache off (not enough VRAM)"
    kwargs = dict(input_prompt=prompt, n_prompt=None if turbo else negative_prompt or None, seed=seed, width=width,
                  height=height, sampling_steps=spec["steps"] or int(steps), sample_solver=spec["solver"],
                  guide_scale=1.0 if turbo else float(guidance), batch_size=1, callback=callback,
                  VAE_tile_size=(True, VAE_TILE), loras_slists=None, input_ref_images=ref_images or None,
                  outpainting_dims=outpainting_dims,
                  custom_settings={"qwen21_kv_cache": "Enabled" if use_kv else "Disabled",
                                   "rgba": "Enabled" if rgba else "Disabled"})
    if edit_image is not None:
        # Wan2GP's inpainting inputs: control frame [3, 1, H, W] in [-1, 1], mask [1, 1, H, W] in [0, 1]
        frame = torch.from_numpy(np.array(edit_image.convert("RGB"), copy=True)).permute(2, 0, 1).float()
        mask = torch.from_numpy(np.array(edit_mask.convert("L"), copy=True)).float().div_(255)
        kwargs.update(input_frames=frame.div_(127.5).sub_(1).unsqueeze(1), input_masks=mask[None, None],
                      denoising_strength=float(strength), masking_strength=1.0, model_mode=0)
    try:
        return engine.pipeline.generate(**kwargs)
    except torch.OutOfMemoryError:
        if not use_kv:
            raise
        print("⚠️ Out of VRAM with the KV cache - retrying without it.")
        gc.collect(); torch.cuda.empty_cache()
        kwargs["custom_settings"] = {**kwargs["custom_settings"], "qwen21_kv_cache": "Disabled"}
        PHASE_TIMES["kv"] = "KV cache off (retried)"
        return engine.pipeline.generate(**kwargs)
    except RuntimeError as e:
        if "non-finite" not in str(e) or int8_backend._backend == "pytorch":
            raise
        print("⚠️ INT8 kernels produced non-finite values - switching to the exact INT8 path and retrying.")
        int8_backend.configure("disabled", 1)
        return engine.pipeline.generate(**kwargs)


def fp16_report():
    """One line on FP16 headroom for the last image ('' when the transformer runs in BF16)."""
    if "fp16" not in ACT_PEAK:
        return ""
    peak = float(ACT_PEAK["fp16"])
    if peak >= FP16_MAX * 0.99:
        return f"⚠️ FP16 overflow (activations hit the ±{FP16_MAX:.0f} limit): quality may suffer."
    return f"FP16 headroom OK (peak {peak:.0f} of {FP16_MAX:.0f})"


# ==== PROMPT HELPERS ====
STYLE_MODIFIERS = {
    "Cinematic": "cinematic lighting, dramatic shadows, film grain, highly detailed, masterpiece",
    "Photographic": "professional photography, 35mm lens, f/2.8, depth of field, natural lighting, photorealistic, highly detailed",
    "Anime": "anime key visual, vibrant color palette, clean line art, highly detailed digital illustration",
    "Cyberpunk": "cyberpunk aesthetic, glowing neon lights, futuristic city, volumetric smoke, high contrast",
    "Fantasy": "mythical fantasy scene, glowing magical particles, ethereal light, digital painting, masterpiece",
}


TRANSPARENCY_PREFIX = "This is an RGBA image with transparency."
TRANSPARENCY_SUFFIX = "The image has an alpha channel and the background is transparent."


def apply_transparency(prompt):
    # Wan2GP's guidance: RGBA output only comes out transparent when the prompt asks for an RGBA image, an alpha
    # channel and a transparent background, so add whatever the prompt doesn't already say
    text = prompt.lower()
    if "rgba" not in text:
        prompt = f"{TRANSPARENCY_PREFIX} {prompt}"
    if "transparent" not in text or "alpha" not in text:
        prompt = f"{prompt.rstrip().rstrip('.')}. {TRANSPARENCY_SUFFIX}"
    return prompt


def apply_style(prompt, style_preset):
    modifier = STYLE_MODIFIERS.get(style_preset)
    if not modifier or modifier in prompt:
        return prompt
    return f"{prompt}, {modifier}" if prompt else modifier


# ==== GENERATION FUNCTION ====
TASK_CREATE = "🖼️ Create (text → image)"
TASK_REFS = "🧩 Edit & Multi-Reference"
TASK_INPAINT = "🖌️ Inpaint (paint the area to change)"
TASK_OUTPAINT = "🔲 Outpaint (extend the image)"
OUTPAINT_RATIOS = {"Custom margins (sliders)": "", "16:9 Landscape": "16:9", "9:16 Portrait": "9:16", "1:1 Square": "1:1",
                   "4:3 Standard": "4:3", "3:4 Portrait": "3:4", "2:3 Poster": "2:3", "21:9 Cinematic": "21:9"}
OUTPAINT_DEFAULT_PROMPT = "Continue the scene naturally beyond the original borders, matching its style and lighting."
MODE_VIGGLE = "✨ Turbo HQ · Viggle v0.2.1 · 6 steps (best fast quality)"
MODE_PRUNA = "⚡ Turbo · Pruna v0.1 · 8 steps"
MODE_ALIBABA = "🚀 Turbo Fast · Alibaba Fun-Acc · 4 steps (fastest)"
MODE_QUALITY = "🎨 Quality · base model + CFG (slow, follows prompts best)"
# Each turbo LoRA only works with the sigma schedule it was distilled on: Viggle [1, 0.9375, 0.875, 0.75, 0.5, 0.25]
# (Wan2GP's viggle_v02, resolution-shifted), Pruna's and Alibaba's fixed grids (Wan2GP's fixed-sigma "pruna" path:
# Pruna's 8-step grid is built in, Alibaba's 4-step grid from pdd_config.json is registered by load_alibaba_pdd).
# No CFG, strength 1.0.
MODES = {MODE_VIGGLE: {"lora": "viggle", "steps": 6, "solver": "viggle_v02"},
         MODE_PRUNA: {"lora": "pruna", "steps": 8, "solver": "pruna"},
         MODE_ALIBABA: {"lora": "alibaba", "steps": 4, "solver": "pruna"},
         MODE_QUALITY: {"lora": None, "steps": None, "solver": "default"}}
MATCH_INPUT = "Match input image"
ASPECT_RATIOS = {MATCH_INPUT: None, "1:1 Square": (1, 1), "16:9 Landscape": (16, 9), "9:16 Portrait": (9, 16),
                 "4:3 Standard": (4, 3), "3:4 Portrait": (3, 4), "2:3 Poster": (2, 3)}
RESOLUTIONS = {"768px (fastest)": 768, "1024px (recommended)": 1024, "1536px (high, slow on T4)": 1536}
MAX_REFS = 10
BLOCK = 32  # Qwen-Image 2.1 sizes must be multiples of 32
ENGINE_READY = threading.Event()
if getattr(globals().get("QWEN_ENGINE"), "dtype", None) == DTYPE:  # older notebook versions: reload
    ENGINE_READY.set()


def resolve_size(aspect_ratio, resolution, input_image=None):
    # Longest side = base resolution (or the input image's shape at that pixel budget), multiples of 32
    base = RESOLUTIONS.get(resolution, 1024)
    ratio = ASPECT_RATIOS.get(aspect_ratio)
    if ratio is None:
        if input_image is None:
            ratio = (1, 1)
        else:
            ratio = input_image.size
            scale = base / max(ratio)
            return (max(256, int(ratio[0] * scale) // BLOCK * BLOCK), max(256, int(ratio[1] * scale) // BLOCK * BLOCK))
    rw, rh = ratio
    if rw >= rh:
        return base, max(256, int(base * rh / rw) // BLOCK * BLOCK)
    return max(256, int(base * rw / rh) // BLOCK * BLOCK), base


def fit_reference(image, width, height):
    # Wan2GP's reference preprocessing for this model: same pixel count as the output, own aspect ratio kept.
    # Transparency is kept: the VAE encodes RGBA, so cut-outs and stickers stay transparent references.
    has_alpha = image.mode in ("RGBA", "LA") or (image.mode == "P" and "transparency" in image.info)
    image = image.convert("RGBA" if has_alpha else "RGB")
    scale = (width * height / (image.width * image.height)) ** 0.5
    size = (max(BLOCK, round(image.width * scale / BLOCK) * BLOCK), max(BLOCK, round(image.height * scale / BLOCK) * BLOCK))
    return image if image.size == size else image.resize(size, Image.Resampling.LANCZOS)


def gallery_images(value):
    # gr.Gallery gives [(image, caption), ...]; images may arrive as PIL images or file paths
    images = []
    for item in value or []:
        image = item[0] if isinstance(item, (list, tuple)) else item
        images.append(image if isinstance(image, Image.Image) else Image.open(image))
    return images


def editor_image_and_mask(value):
    # gr.ImageEditor gives {"background", "layers", "composite"}: painted strokes (any layer) become the mask
    if not value or value.get("background") is None:
        return None, None
    background = value["background"]
    background = background if isinstance(background, Image.Image) else Image.open(background)
    mask = np.zeros((background.height, background.width), dtype=bool)
    for layer in value.get("layers") or []:
        layer = layer if isinstance(layer, Image.Image) else Image.open(layer)
        alpha = np.array(layer.convert("RGBA").resize(background.size))[..., 3]
        mask |= alpha > 0
    return background.convert("RGB"), Image.fromarray(mask.astype(np.uint8) * 255)


def outpaint_layout(image, resolution, ratio_label, top, bottom, left, right):
    """Canvas for Wan2GP's Red Canvas outpainting. Margins are % of the source (top, bottom, left, right); with a
    target ratio they only weight where the padding goes. Geometry comes from Wan2GP's own helpers, so the source
    lands exactly where the pipeline crops it back out (32 px aligned). Returns (canvas, dims, width, height)."""
    from shared.utils.utils import get_outpainting_frame_location, get_outpainting_full_area_dimensions, resolve_outpainting_dims
    image = image.convert("RGB")
    dims = resolve_outpainting_dims(image.height, image.width, [top, bottom, left, right], OUTPAINT_RATIOS.get(ratio_label, ""))
    if not any(dims):
        return None, None, None, None
    full_height, full_width = get_outpainting_full_area_dimensions(image.height, image.width, dims)
    scale = RESOLUTIONS.get(resolution, 1024) / max(full_width, full_height)
    width = max(256, round(full_width * scale / BLOCK) * BLOCK)
    height = max(256, round(full_height * scale / BLOCK) * BLOCK)
    source_height, source_width, top_px, left_px = get_outpainting_frame_location(height, width, dims, 1, quantize_margins=BLOCK)
    canvas = Image.new("RGB", (width, height), (255, 0, 0))  # red = area to fill (the pipeline redraws it the same way)
    canvas.paste(image.resize((source_width, source_height), Image.Resampling.LANCZOS), (left_px, top_px))
    return canvas, dims, width, height


def outpaint_preview(image, resolution, ratio_label, top, bottom, left, right):
    if image is None:
        return None
    canvas, _, _, _ = outpaint_layout(image, resolution, ratio_label, top, bottom, left, right)
    return canvas


def stop_generation():
    if ENGINE_READY.is_set():
        QWEN_ENGINE.pipeline._interrupt = True
    return "⏹️ Stopping after the current step..."


# ==== LIVE PROGRESS FOR THE CLOUDFLARE LINK ====
# Gradio streams progress over Server-Sent Events, which Cloudflare quick tunnels hold back until the request ends,
# so on the trycloudflare link the page polls this small JSON endpoint instead (plain requests pass through fine).
PROGRESS = {"active": False, "fraction": 0.0, "text": "", "started": 0.0}


def report_progress(fraction, text):
    PROGRESS.update(active=True, fraction=float(fraction), text=text)


PROGRESS_HEAD = """
<script>
(function () {
  if (!/\\.trycloudflare\\.com$/.test(location.hostname)) return;  // Gradio's own bar works on the other links
  var box = null, fill = null, label = null, hideAt = 0;
  function build() {
    box = document.createElement("div");
    box.style.cssText = "position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:99999;width:min(560px,92vw);" +
      "background:#1f2937;border-radius:12px;box-shadow:0 8px 24px rgba(0,0,0,.25);padding:10px 14px;display:none;" +
      "font-family:Inter,sans-serif;color:#fff;font-size:13px";
    label = document.createElement("div");
    label.style.cssText = "margin-bottom:6px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis";
    var track = document.createElement("div");
    track.style.cssText = "height:8px;background:rgba(255,255,255,.18);border-radius:6px;overflow:hidden";
    fill = document.createElement("div");
    fill.style.cssText = "height:100%;width:0;background:linear-gradient(135deg,#667eea 0%,#764ba2 100%);transition:width .6s ease";
    track.appendChild(fill); box.appendChild(label); box.appendChild(track); document.body.appendChild(box);
  }
  async function tick() {
    try {
      var r = await fetch("aiq_progress?t=" + Date.now(), {cache: "no-store"});
      var p = await r.json();
      if (!box && document.body) build();
      if (box) {
        if (p.active) {
          box.style.display = "block"; hideAt = 0;
          fill.style.width = Math.round(p.fraction * 100) + "%";
          label.textContent = "⏳ " + p.text + " · " + Math.round(p.elapsed) + "s";
        } else if (box.style.display === "block") {
          if (!hideAt) { fill.style.width = "100%"; label.textContent = "✅ Done in " + Math.round(p.elapsed) + "s"; hideAt = Date.now() + 4000; }
          else if (Date.now() > hideAt) { box.style.display = "none"; }
        }
      }
    } catch (e) {}
    setTimeout(tick, 1000);
  }
  tick();
})();
</script>
"""


def generate_image(task, prompt, refs_value, editor_value, strength, outpaint_image, outpaint_ratio, out_top, out_bottom,
                   out_left, out_right, negative_prompt, style_preset, mode, transparent, steps, guidance, aspect_ratio,
                   resolution, seed, num_images, progress=gr.Progress()):
    PROGRESS.update(active=True, fraction=0.0, text="Starting...", started=time.time())
    while not ENGINE_READY.is_set():  # clicked while Step 2 is still loading the model
        report_progress(0.0, "The model is still loading...")
        if LOAD_ERROR:
            PROGRESS["active"] = False
            yield gr.update(), f"❌ Model failed to load: {LOAD_ERROR}", gr.update(visible=False)
            return
        yield gr.update(), "⏳ The model is still loading (about a minute after the links appear)...", gr.update(visible=False)
        time.sleep(2)
    pipeline = QWEN_ENGINE.pipeline
    try:
        pipeline._interrupt = False
        prompt = apply_style((prompt or "").strip(), style_preset)
        if not prompt and task == TASK_OUTPAINT:
            prompt = OUTPAINT_DEFAULT_PROMPT
        if not prompt:
            yield gr.update(), "❌ Please enter a prompt.", gr.update(visible=False)
            return
        if transparent:
            prompt = apply_transparency(prompt)

        # ---- inputs for the selected task ----
        refs, edit_image, edit_mask, outpainting_dims = [], None, None, None
        if task == TASK_OUTPAINT:
            if outpaint_image is None:
                yield gr.update(), "❌ Upload the image to extend.", gr.update(visible=False)
                return
            canvas, outpainting_dims, width, height = outpaint_layout(outpaint_image, resolution, outpaint_ratio,
                                                                      out_top, out_bottom, out_left, out_right)
            if canvas is None:
                yield gr.update(), "❌ Nothing to extend: raise a margin slider or pick a wider/taller target ratio.", gr.update(visible=False)
                return
            refs = [canvas]
        elif task == TASK_REFS:
            refs = gallery_images(refs_value)
            if not refs:
                yield gr.update(), "❌ Upload at least one image (up to 10).", gr.update(visible=False)
                return
            if len(refs) > MAX_REFS:
                yield gr.update(), f"❌ Qwen-Image 2.1 accepts at most {MAX_REFS} reference images.", gr.update(visible=False)
                return
            width, height = resolve_size(aspect_ratio, resolution, refs[0])
            refs = [fit_reference(image, width, height) for image in refs]
        elif task == TASK_INPAINT:
            source, mask = editor_image_and_mask(editor_value)
            if source is None:
                yield gr.update(), "❌ Upload an image, then paint over the area to change.", gr.update(visible=False)
                return
            if not np.array(mask).any():
                yield gr.update(), "❌ Paint over the area you want to change (the brush strokes are the mask).", gr.update(visible=False)
                return
            width, height = resolve_size(MATCH_INPUT, resolution, source)  # keeps the image's own shape
            edit_image = source.resize((width, height), Image.Resampling.LANCZOS)
            edit_mask = mask.resize((width, height), Image.Resampling.NEAREST)
        else:
            width, height = resolve_size(aspect_ratio, resolution)

        spec = MODES[mode]
        turbo = spec["lora"] is not None
        if turbo and QWEN_ENGINE.adapters.get(spec["lora"]) is None:
            yield gr.update(), "❌ This turbo LoRA did not load (see the Step 2 log). Pick another mode.", gr.update(visible=False)
            return
        negative_prompt = "" if turbo else (negative_prompt or "").strip()
        total_steps = spec["steps"] or int(steps)
        num_images = int(num_images)
        initial_seed = random.randint(0, 2**32 - 1) if seed is None or seed < 0 else int(seed)

        yield gr.update(value=None, selected_index=None), "⏳ Initializing generation...", gr.update(visible=False)

        paths, seeds_used, start_time, headroom = [], [], time.time(), ""
        for idx in range(num_images):
            current_seed = (initial_seed + idx) % (2**32)

            def cb(step_idx, latent=None, is_start=False, override_num_inference_steps=None, progress_title=None, **kwargs):
                if progress_title:  # text encoder layers / VAE tiles
                    total = override_num_inference_steps or 1
                    done = 0.98 if "VAE" in progress_title else 0.0  # the VAE runs after the last step
                    fraction, desc = min((idx + done) / num_images, 0.99), f"Image {idx+1}/{num_images} - {progress_title} {max(step_idx + 1, 0)}/{total}"
                elif step_idx >= 0:
                    fraction, desc = min((idx + (step_idx + 1) / total_steps) / num_images, 0.99), f"Image {idx+1}/{num_images} - Step {step_idx+1}/{total_steps}"
                else:
                    return
                progress(fraction, desc=desc)
                report_progress(fraction, desc)

            inputs = (f", outpaint margins top/bottom/left/right {[round(v) for v in outpainting_dims]}%" if outpainting_dims else
                      f", {len(refs)} reference image(s) {[image.size for image in refs]}" if refs else
                      f", masked edit ({int(np.array(edit_mask).astype(bool).mean() * 100)}% of the image, strength {strength})"
                      if edit_image is not None else "")
            print(f"Generating image {idx+1}/{num_images}: {width}x{height}, {spec['lora'] or 'quality'}, "
                  f"steps={total_steps}, seed={current_seed}{inputs}\n  prompt: {prompt}")
            t_img = time.time()
            result = run_generation(prompt, negative_prompt, mode, steps, guidance, width, height, current_seed,
                                    callback=cb, ref_images=refs, edit_image=edit_image, edit_mask=edit_mask,
                                    strength=strength, outpainting_dims=outpainting_dims, rgba=transparent)
            if result is None:
                if pipeline._interrupt:
                    break
                raise RuntimeError(f"Generation of image {idx+1} failed (returned None).")

            # Output tensor [3, 1, H, W] uint8 on CPU -> PNG in the output folder
            img_path = os.path.join(OUT_DIR, f"qwen21_{time.strftime('%Y%m%d_%H%M%S')}_seed{current_seed}.png")
            # [3 or 4, 1, H, W] uint8: 4 channels = RGBA, saved as a transparent PNG
            Image.fromarray(result[:, 0].permute(1, 2, 0).contiguous().numpy(),
                            mode="RGBA" if result.shape[0] == 4 else "RGB").save(img_path)
            paths.append(img_path)
            seeds_used.append(current_seed)
            took = time.time() - t_img
            text_t, vae_t = PHASE_TIMES.get("text", 0.0), PHASE_TIMES.get("vae", 0.0)
            headroom = fp16_report()
            print(f"  ✅ {took:.1f}s (text {text_t:.1f}s · denoise {took - text_t - vae_t:.1f}s · VAE {vae_t:.1f}s · "
                  f"{PHASE_TIMES.get('kv', 'KV cache off')}) -> {img_path}"
                  + (f"\n  {headroom}" if headroom else ""))
            yield gr.update(value=paths, selected_index=0), f"⏳ Generated {len(paths)}/{num_images} images...", gr.update(visible=False)

        if not paths:
            yield gr.update(value=None, selected_index=None), "⏹️ Generation stopped.", gr.update(visible=False)
            return

        zip_path = None
        if len(paths) > 1:
            zip_path = os.path.join(OUT_DIR, f"qwen21_batch_{time.strftime('%Y%m%d_%H%M%S')}.zip")
            with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as zipf:  # PNGs are already compressed
                for path in paths:
                    zipf.write(path, os.path.basename(path))

        elapsed = time.time() - start_time
        stopped = " (stopped early)" if pipeline._interrupt else ""
        status = (f"✅ Generated {len(paths)} image(s) in {elapsed:.1f}s ({elapsed / len(paths):.1f}s/image){stopped} | "
                  f"Seeds: {seeds_used} | {width}x{height} | {total_steps} steps" + (f" | {headroom}" if headroom else ""))
        yield gr.update(value=paths, selected_index=0), status, gr.update(value=zip_path, visible=zip_path is not None)
    except Exception as e:
        traceback.print_exc()
        yield gr.update(value=None, selected_index=None), f"❌ Error: {str(e)}", gr.update(visible=False)
    finally:
        pipeline._interrupt = False
        PROGRESS.update(active=False, fraction=1.0)
        gc.collect()


def switch_mode(mode):
    quality = mode == MODE_QUALITY
    return gr.update(interactive=quality), gr.update(interactive=quality), gr.update(interactive=quality)


TASK_TIPS = {
    TASK_CREATE: ('Describe what is visible (subjects, layout, colors, lighting). Put exact text in "double quotes" '
                  "and say which language it is in (Qwen-Image tends to invent Chinese lettering otherwise)."),
    TASK_REFS: ("One image: describe the edit, e.g. \"Change the jacket to red leather, keep everything else\". "
                "Several: refer to them in upload order, e.g. \"Put the person from image 1 in the room from image 2\"."),
    TASK_INPAINT: 'Describe what belongs in the painted area, e.g. "a golden retriever sitting on the grass".',
    TASK_OUTPAINT: ("Optional: describe what the wider view contains, e.g. \"a busy harbor with boats\". "
                    "Left empty, the scene is continued naturally. The red area in the preview gets filled."),
}


def switch_task(task):
    return (gr.update(visible=task == TASK_REFS), gr.update(visible=task == TASK_INPAINT),
            gr.update(visible=task == TASK_INPAINT), gr.update(visible=task == TASK_OUTPAINT),
            gr.update(info=TASK_TIPS[task]),
            gr.update(value=MATCH_INPUT if task != TASK_CREATE else "1:1 Square",
                      interactive=task not in (TASK_INPAINT, TASK_OUTPAINT)))


# ==== GRADIO UI (AIQUEST BRANDED) ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
.status-line { text-align: center; color: #6b7280; font-size: 13px; margin: -8px 0 12px 0; }
.footer { text-align: center; padding: 20px; margin-top: 30px; border-top: 2px solid #e5e7eb; color: #6b7280; font-size: 13px; }
.footer a { color: #764ba2; text-decoration: none; margin: 0 8px; font-weight: 600; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">⚡ Qwen-Image 2.1 Uncensored</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Create · Edit · Multi-Reference · Inpaint · INT8 Tensor Cores</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

STATUS_HTML = (f'<div class="status-line">🖥️ {gpu.name} · transformer {_dt(DTYPE)} · '
               f'INT8 tensor-core kernels (Comfy Kitchen) · Attention: SDPA</div>')

FOOTER_HTML = """
<div class="footer">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · Qwen-Image 2.1 Uncensored · Wan2GP INT8 Edition<br>
  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank">YouTube</a> |
  <a href="https://x.com/aiquestacademy" target="_blank">X (Twitter)</a> |
  <a href="https://aiquest.site" target="_blank">aiquest.site</a>
</div>
"""

with gr.Blocks(theme=gr.themes.Soft(), css=CSS, head=PROGRESS_HEAD, title="Qwen-Image 2.1 Uncensored | AIQUEST") as demo:
    gr.HTML(BRAND_HTML)
    gr.HTML(STATUS_HTML)

    with gr.Row():
        with gr.Column(scale=1):
            task_in = gr.Radio(label="🧭 Task", choices=[TASK_CREATE, TASK_REFS, TASK_INPAINT, TASK_OUTPAINT], value=TASK_CREATE)
            refs_in = gr.Gallery(label="🧩 Images (upload order = image 1, image 2, ... up to 10)", type="pil",
                                 interactive=True, columns=4, height=220, visible=False)
            editor_in = gr.ImageEditor(label="🖌️ Upload an image, then paint over the area to change", type="pil",
                                       sources=["upload", "clipboard"], layers=False, visible=False,
                                       brush=gr.Brush(colors=["#FF0000"], color_mode="fixed", default_size=40))
            strength_in = gr.Slider(label="💪 Change strength (Inpaint)", minimum=0.3, maximum=1.0, step=0.05, value=1.0,
                                    visible=False, info="1.0 fully regenerates the painted area; lower keeps more of the original.")
            with gr.Group(visible=False) as outpaint_group:
                with gr.Row():
                    outpaint_in = gr.Image(label="🔲 Image to extend", type="pil", sources=["upload", "clipboard"], height=240)
                    outpaint_preview_out = gr.Image(label="👁️ Canvas preview (red = new area)", interactive=False, height=240)
                outpaint_ratio_in = gr.Dropdown(label="📐 Expand to", choices=list(OUTPAINT_RATIOS),
                                                value="Custom margins (sliders)",
                                                info="With a target ratio, the sliders only decide where the new space goes "
                                                     "(equal = centered, 0 = keep that edge).")
                with gr.Row():
                    out_left_in = gr.Slider(label="⬅️ Left %", minimum=0, maximum=100, step=5, value=25)
                    out_right_in = gr.Slider(label="➡️ Right %", minimum=0, maximum=100, step=5, value=25)
                with gr.Row():
                    out_top_in = gr.Slider(label="⬆️ Top %", minimum=0, maximum=100, step=5, value=0)
                    out_bottom_in = gr.Slider(label="⬇️ Bottom %", minimum=0, maximum=100, step=5, value=0)
            prompt_in = gr.Textbox(label="📝 Prompt", lines=4, info=TASK_TIPS[TASK_CREATE],
                                   placeholder='A vertical poster divided into four panels, titled "THE SEARCH"...')
            mode_in = gr.Radio(
                label="🚀 Mode", choices=[MODE_VIGGLE, MODE_PRUNA, MODE_ALIBABA, MODE_QUALITY], value=MODE_VIGGLE,
                info="Three distilled turbo LoRAs (no CFG, no negative prompt): Viggle 6 steps, Pruna 8 steps, "
                     "Alibaba 4 steps. Results differ by subject, so try the same seed in each. "
                     "Quality: 40-step base model with CFG (~7-10x slower).")
            transparent_in = gr.Checkbox(
                label="🫥 Transparent background (RGBA PNG)", value=False,
                info="Stickers, logos, cut-outs, or extracting a subject from an uploaded image. Adds the transparency "
                     "wording to the prompt and saves a PNG with an alpha channel.")
            with gr.Row():
                style_in = gr.Dropdown(label="🎨 Style Preset", choices=["None"] + list(STYLE_MODIFIERS), value="None")
                resolution_in = gr.Dropdown(label="🖥️ Base Resolution", choices=list(RESOLUTIONS),
                                            value="1024px (recommended)", info="Longest side.")
            with gr.Row():
                aspect_in = gr.Dropdown(label="📏 Aspect Ratio", choices=list(ASPECT_RATIOS), value="1:1 Square")
                num_images_in = gr.Slider(label="🖼️ Number of Images", minimum=1, maximum=4, step=1, value=1,
                                          info="One after another (seed +1 each).")

            with gr.Accordion("⚙️ Quality Mode & Advanced Settings", open=False):
                negative_in = gr.Textbox(label="🚫 Negative Prompt (Quality mode)", lines=2, interactive=False,
                                         placeholder="low quality, blurry, distorted, bad anatomy, watermark, chinese text")
                with gr.Row():
                    steps_in = gr.Slider(label="🔁 Steps (Quality mode)", minimum=10, maximum=50, step=1, value=40,
                                         interactive=False, info="40 recommended.")
                    guidance_in = gr.Slider(label="🧲 Guidance / CFG (Quality mode)", minimum=1.0, maximum=8.0,
                                            step=0.5, value=4.0, interactive=False, info="4 recommended.")
                seed_in = gr.Number(label="🎲 Seed (-1 for Random)", value=-1, precision=0)

            with gr.Row():
                gen_btn = gr.Button("🎨 Generate", variant="primary", size="lg")
                stop_btn = gr.Button("⏹️ Stop", variant="stop", size="lg")

        with gr.Column(scale=1):
            gallery_out = gr.Gallery(label="🖼️ Generated Images", columns=2, rows=2, object_fit="contain", preview=True)
            zip_out = gr.File(label="📦 Download All Images (ZIP)", visible=False)
            status_out = gr.Textbox(label="ℹ️ Status", interactive=False)
            gr.Markdown("**T4 speed tip:** every reference image adds as many tokens as the output image, so "
                        "steps slow down quickly: roughly 2-3x with 1 reference and 5-8x with 3. "
                        "Use 768px for quick multi-reference tests.")

    gr.HTML(FOOTER_HTML)

    task_in.change(switch_task, inputs=task_in,
                   outputs=[refs_in, editor_in, strength_in, outpaint_group, prompt_in, aspect_in], queue=False)
    outpaint_inputs = [outpaint_in, resolution_in, outpaint_ratio_in, out_top_in, out_bottom_in, out_left_in, out_right_in]
    for component in outpaint_inputs:
        component.change(outpaint_preview, inputs=outpaint_inputs, outputs=outpaint_preview_out, queue=False,
                         show_progress="hidden")
    mode_in.change(switch_mode, inputs=mode_in, outputs=[negative_in, steps_in, guidance_in], queue=False)
    gen_btn.click(
        generate_image,
        inputs=[task_in, prompt_in, refs_in, editor_in, strength_in, outpaint_in, outpaint_ratio_in, out_top_in,
                out_bottom_in, out_left_in, out_right_in, negative_in, style_in, mode_in, transparent_in, steps_in, guidance_in, aspect_in, resolution_in, seed_in, num_images_in],
        outputs=[gallery_out, status_out, zip_out],
        concurrency_limit=1,
    )
    stop_btn.click(stop_generation, outputs=[status_out], queue=False)


# ==== LAUNCH: Gradio share link + Cloudflare quick tunnel (backup when the Gradio link fails) ====
SERVER_PORT = 7860


def start_cloudflare_tunnel(port):
    found = {"url": None}
    binary = "/tmp/cloudflared"
    try:
        if not os.path.exists(binary):
            urllib.request.urlretrieve(
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", binary)
            os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
        subprocess.run(["pkill", "-f", "cloudflared tunnel"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        tunnel = subprocess.Popen([binary, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    except Exception as e:
        print(f"⚠️ Cloudflare tunnel unavailable: {e}")
        return found

    def _watch():
        for line in tunnel.stdout:  # keep draining so cloudflared never blocks on a full pipe
            match = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", line)  # skip cloudflared's API host
            if match and not found["url"]:
                found["url"] = match.group(0)

    threading.Thread(target=_watch, daemon=True).start()
    return found


_prev_demo = globals().get("demo_prev")  # re-running this cell: free the port held by the previous UI
if _prev_demo is not None:
    try:
        _prev_demo.close()
    except Exception:
        pass
demo_prev = demo
LOAD_ERROR = None

print("=" * 65)
print("🎨 Qwen-Image 2.1 Uncensored - Create · Edit · Multi-Reference | AIQUEST Academy")
print(f"⚡ GPU: {gpu.name} ({_gb(gpu.total_memory)} VRAM, SM {gpu.major}.{gpu.minor}) | "
      f"RAM: {_gb(psutil.virtual_memory().total)} | transformer {_dt(DTYPE)}")
print("=" * 65)
cf = start_cloudflare_tunnel(SERVER_PORT)  # connects in the background while the model loads

# ==== LOAD THE MODEL (the links are printed once everything is ready) ====
if ENGINE_READY.is_set():
    print("♻️ Reusing the model loaded by the previous run of this cell.")
else:
    _t0 = time.time()
    QWEN_ENGINE = _load_engine()
    gc.collect(); torch.cuda.empty_cache()
    ENGINE_READY.set()
    print(f"✅ Model ready in {time.time() - _t0:.0f}s | INT8 kernels: {INT8_LABELS[int8_backend._backend]} | "
          f"VRAM in use: {_gb(torch.cuda.memory_allocated())} | RAM in use: {_gb(psutil.virtual_memory().used)}")

# ==== LAUNCH THE UI ====
demo.queue(max_size=20)
with contextlib.redirect_stdout(io.StringIO()):  # Gradio's own "Running on public URL" line; printed below instead
    demo.launch(server_name="127.0.0.1", server_port=SERVER_PORT, share=True, inline=False, show_error=True,
                prevent_thread_lock=True, quiet=True, ssr_mode=False, allowed_paths=[OUT_DIR])
from fastapi import Request
from fastapi.responses import JSONResponse, FileResponse
from fastapi.routing import APIRoute

# ==== n8n API: prompt + refs + settings -> generated PNG URL ====
N8N_API_LOCK = threading.Lock()

def _api_bool(value, default=False):
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in ("1", "true", "yes", "on")

def _api_mode(value):
    if value is None:
        return MODE_VIGGLE

    mode = str(value).strip()

    aliases = {
        "viggle": MODE_VIGGLE,
        "pruna": MODE_PRUNA,
        "alibaba": MODE_ALIBABA,
        "quality": MODE_QUALITY,
    }

    return aliases.get(mode.lower(), mode if mode in MODES else MODE_VIGGLE)

def _download_api_image(url, width, height):
    req = urllib.request.Request(
        str(url),
        headers={"User-Agent": "Mozilla/5.0"}
    )

    with urllib.request.urlopen(req, timeout=60) as response:
        image_bytes = response.read()

    image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    return fit_reference(image, width, height)

def _n8n_generate(payload, request):
    if not ENGINE_READY.is_set():
        return JSONResponse(
            {"error": "Model is not ready yet."},
            status_code=503
        )

    prompt = str(payload.get("prompt") or "").strip()

    if not prompt:
        return JSONResponse(
            {"error": "prompt is required."},
            status_code=400
        )

    # ---------------------------------
    # Output size
    # ---------------------------------
    if payload.get("width") is not None and payload.get("height") is not None:
        width = int(payload["width"])
        height = int(payload["height"])
    else:
        width, height = resolve_size(
            payload.get("aspect_ratio", "16:9 Landscape"),
            payload.get("resolution", "1024px (recommended)")
        )

    if width < 256 or height < 256:
        return JSONResponse(
            {"error": "width and height must be at least 256."},
            status_code=400
        )

    # Qwen-Image 2.1 requires 32px-aligned dimensions.
    width = (width // BLOCK) * BLOCK
    height = (height // BLOCK) * BLOCK

    # ---------------------------------
    # Generation settings
    # ---------------------------------
    mode = _api_mode(payload.get("mode"))
    steps = int(payload.get("steps", 40))
    guidance = float(payload.get("guidance", 4.0))
    negative_prompt = str(payload.get("negative_prompt") or "")
    transparent = _api_bool(payload.get("transparent"), False)
    style_preset = payload.get("style_preset", "None")

    seed_value = payload.get("seed", -1)
    try:
        seed_value = int(seed_value)
    except Exception:
        seed_value = -1

    seed = (
        random.randint(0, 2**32 - 1)
        if seed_value < 0
        else seed_value
    )

    # ---------------------------------
    # Reference images
    # Supports:
    #   ref_image
    #   ref_image2
    #   ref_images: [...]
    # ---------------------------------
    ref_urls = []

    for key in ("ref_image", "ref_image2"):
        if payload.get(key):
            ref_urls.append(payload[key])

    if isinstance(payload.get("ref_images"), list):
        ref_urls.extend(
            [url for url in payload["ref_images"] if url]
        )

    if len(ref_urls) > MAX_REFS:
        return JSONResponse(
            {"error": f"Maximum {MAX_REFS} reference images allowed."},
            status_code=400
        )

    refs = []

    try:
        for url in ref_urls:
            refs.append(_download_api_image(url, width, height))
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(
            {"error": f"Reference image download failed: {e}"},
            status_code=400
        )

    # ---------------------------------
    # Apply style / transparency
    # ---------------------------------
    prompt = apply_style(
        prompt,
        style_preset
    )

    if transparent:
        prompt = apply_transparency(prompt)

    # ---------------------------------
    # Generate
    # ---------------------------------
    with N8N_API_LOCK:
        try:
            result = run_generation(
                prompt=prompt,
                negative_prompt=negative_prompt,
                mode=mode,
                steps=steps,
                guidance=guidance,
                width=width,
                height=height,
                seed=seed,
                ref_images=refs,
                edit_image=None,
                edit_mask=None,
                strength=1.0,
                outpainting_dims=None,
                rgba=transparent
            )

            if result is None:
                return JSONResponse(
                    {"error": "Image generation failed."},
                    status_code=500
                )

            img_path = os.path.join(
                OUT_DIR,
                f"n8n_qwen21_{time.strftime('%Y%m%d_%H%M%S')}_seed{seed}.png"
            )

            Image.fromarray(
                result[:, 0]
                .permute(1, 2, 0)
                .contiguous()
                .cpu()
                .numpy(),
                mode="RGBA" if result.shape[0] == 4 else "RGB"
            ).save(img_path)

            filename = os.path.basename(img_path)

            # Public URL that n8n can store in Airtable and later use
            # for the animation/video stage.
            base_url = str(request.base_url).rstrip("/")
            image_url = f"{base_url}/outputs/{filename}"

            return JSONResponse({
                "image_url": image_url,
                "filename": filename,
                "width": width,
                "height": height,
                "seed": seed,
                "mode": mode,
                "reference_images": len(refs)
            })

        except Exception as e:
            traceback.print_exc()
            return JSONResponse(
                {"error": str(e)},
                status_code=500
            )

async def _n8n_generate_api(request: Request):
    try:
        data = await request.json()

        if not isinstance(data, dict):
            return JSONResponse(
                {"error": "JSON body must be an object."},
                status_code=400
            )

        return _n8n_generate(data, request)

    except Exception as e:
        traceback.print_exc()
        return JSONResponse(
            {"error": f"Invalid JSON: {e}"},
            status_code=400
        )

# Generate endpoint used by n8n
demo.app.router.add_api_route(
    "/generate",
    _n8n_generate_api,
    methods=["POST"]
)

# Public image route so Airtable / NCA Toolkit can access generated images.
async def _output_file(filename: str):
    safe_name = os.path.basename(filename)
    path = os.path.join(OUT_DIR, safe_name)

    if not os.path.isfile(path):
        return JSONResponse(
            {"error": "Image not found."},
            status_code=404
        )

    return FileResponse(
        path,
        media_type="image/png",
        filename=safe_name
    )

demo.app.router.add_api_route(
    "/outputs/{filename}",
    _output_file,
    methods=["GET"]
)

# Existing progress endpoint
def _progress_route():
    elapsed = time.time() - PROGRESS["started"] if PROGRESS["started"] else 0.0
    return JSONResponse({**PROGRESS, "elapsed": elapsed}, headers={"Cache-Control": "no-store"})
demo.app.router.routes.insert(0, APIRoute("/aiq_progress", _progress_route, methods=["GET"]))  # ahead of Gradio's routes

print("🔌 n8n API: POST /generate")
for _ in range(40):  # give the tunnel up to ~20 s more to report its URL
    if cf["url"]:
        break
    time.sleep(0.5)

print("\n" + "=" * 65)
print("🎨 Qwen-Image 2.1 Uncensored is LIVE!")
print("=" * 65)
print(f"🌐 Cloudflare tunnel: {cf['url'] or '(still connecting, check again in a moment)'}")
print(f"🤖 n8n generate API: {cf['url'] + '/generate' if cf['url'] else '(wait for the Cloudflare URL)'}")
print(f"🔗 Gradio share:      {getattr(demo, 'share_url', None) or '(unavailable, use the Cloudflare link)'}")
print(f"💾 Outputs saved to {OUT_DIR}")
print("=" * 65)
print("🎨 Open a link above and click Generate. Logs for each image appear below.\n")

try:  # keep the cell alive so generation logs keep streaming here
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Generator stopped. Re-run this cell to launch it again (the model stays loaded).")

🎨 Qwen-Image 2.1 Uncensored - Create · Edit · Multi-Reference | AIQUEST Academy
⚡ GPU: Tesla T4 (14.6 GB VRAM, SM 7.5) | RAM: 12.7 GB | transformer FLOAT16
[INT8] Backend: Comfy Kitchen CUDA (setting: auto).
📂 Loading Qwen-Image 2.1 Uncensored (INT8 ConvRot) · transformer FLOAT16...
************ Memory Management for the GPU Poor (mmgp 3.8.2) by DeepBeepMeep ************
Hooked to model 'transformer' (QwenImage21Transformer2DModel)
Hooked to model 'text_encoder' (TextModel)
Async loading plan for model 'text_encoder' : 1472.73 MB will be preloaded (base size of 0.01 MB + 22.2% of recurrent layers data) with a 184.09 MB async circular shuttle
Hooked to model 'vision_encoder' (Qwen3VLVisionModel)
Hooked to model 'vae' (AutoencoderKLQwenImage21)
Lora '/content/qwen21_ckpts/Qwen-Image-2.1-viggle-turbo-v0.2.1-6step-lora-r128.safetensors' was loaded in model 'models.qwen21.transformer'
Lora '/content/qwen21_ckpts/loras/p_qwen_image_2.1_8step_v0.1.safetensors' was loaded in model 'models.qwen

Viggle Turbo Scheduler - Raw Sigmas: [1.0, 0.9375, 0.875, 0.75, 0.5, 0.25]; Applied Sigmas: [1.0, 0.967754, 0.933358, 0.857192, 0.666756, 0.400096, 0.0]; Terminal Shift: None


Denoising: 100%|██████████| 6/6 [00:19<00:00,  3.18s/it]


VAE Decoding - 1 Images


VAE Decoding: 100%|██████████| 1/1 [00:04<00:00,  4.12s/tiles]

⚠️ VAE ran out of VRAM with 1024px tiles - retrying with 512px tiles.


/tmp/ipykernel_1395/862718765.py:940: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(


---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---